# Location Search

In [21]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(".env.development")  # loads TA_API_KEY into os.environ

API_KEY = os.environ.get("TA_API_KEY")
print("Loaded TA_API_KEY?", bool(API_KEY))

if not API_KEY:
    raise RuntimeError("Set TA_API_KEY in your environment (keeps key out of the notebook).")

BASE = "https://api.content.tripadvisor.com/api/v1"
HEADERS = {"accept": "application/json"}

CACHE_DIR = Path("ta_cache")
CACHE_DIR.mkdir(exist_ok=True)

def ta_get(path, params=None, sleep_s=0.25, retries=3):
    """GET wrapper with light retry/backoff + polite pacing."""
    params = dict(params or {})
    params["key"] = API_KEY

    url = f"{BASE}{path}"
    last_err = None

    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * (2 ** i))
                continue
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
            time.sleep(sleep_s)
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (2 ** i))

    raise last_err


Loaded TA_API_KEY? True


In [22]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path

API_KEY = os.environ.get("TA_API_KEY")  # recommended: set env var outside notebook
if not API_KEY:
    raise RuntimeError("Set TA_API_KEY in your environment (keeps key out of the notebook).")

BASE = "https://api.content.tripadvisor.com/api/v1"
HEADERS = {"accept": "application/json"}

CACHE_DIR = Path("ta_cache")
CACHE_DIR.mkdir(exist_ok=True)

def ta_get(path, params=None, sleep_s=0.25, retries=3):
    params = dict(params or {})
    params["key"] = API_KEY
    url = f"{BASE}{path}"

    last_err = None
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * (2 ** i))
                continue
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
            time.sleep(sleep_s)
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (2 ** i))
    raise last_err


In [23]:
seed_places = [
    "Manhattan, New York, USA",
    "Dubai, UAE",
    "Tokyo, Japan",
    "Hong Kong",
    "Singapore",
    "Rio de Janeiro, Brazil",
    "Paris, France",
    "Barcelona, Spain",
    "Rome, Italy",
    "Sydney, Australia",
    "Las Vegas, Nevada, USA",
    "Jerusalem, Israel",
    "London, UK",
    "San Francisco, California, USA",
    "St. Petersburg, Russia",
    "Venice, Italy",
    "Kyoto, Japan",
    "Grand Canyon, Arizona, USA",
    "Sequoia National Park, California, USA",
    "Alaska, USA",

    "Los Angeles, California, USA",
    "Miami, Florida, USA",
    "Orlando, Florida, USA",
    "Chicago, Illinois, USA",
    "Washington, D.C., USA",
    "Boston, Massachusetts, USA",
    "New Orleans, Louisiana, USA",
    "Honolulu, Hawaii, USA",
    "Cancun, Mexico",
    "Mexico City, Mexico",

    "Toronto, Canada",
    "Vancouver, Canada",
    "Montreal, Canada",
    "Havana, Cuba",
    "Buenos Aires, Argentina",
    "Santiago, Chile",
    "Lima, Peru",
    "Machu Picchu, Peru",
    "Bogota, Colombia",
    "Medellin, Colombia",

    "Amsterdam, Netherlands",
    "Brussels, Belgium",
    "Berlin, Germany",
    "Munich, Germany",
    "Vienna, Austria",
    "Prague, Czech Republic",
    "Budapest, Hungary",
    "Zurich, Switzerland",
    "Geneva, Switzerland",
    "Florence, Italy",
    "Milan, Italy",
    "Naples, Italy",
    "Athens, Greece",
    "Santorini, Greece",
    "Mykonos, Greece",
    "Istanbul, Turkey",

    "Cairo, Egypt",
    "Marrakech, Morocco",
    "Cape Town, South Africa",
    "Johannesburg, South Africa",
    "Nairobi, Kenya",

    "Bangkok, Thailand",
    "Phuket, Thailand",
    "Chiang Mai, Thailand",
    "Bali, Indonesia",
    "Jakarta, Indonesia",
    "Kuala Lumpur, Malaysia",
    "Hanoi, Vietnam",
    "Ho Chi Minh City, Vietnam",
    "Seoul, South Korea",
    "Beijing, China",
    "Shanghai, China",
    "Xi'an, China",
    "Taipei, Taiwan",

    "Auckland, New Zealand",
    "Queenstown, New Zealand",
    "Melbourne, Australia",
    "Perth, Australia",
    "Fiji",
]


len(seed_places), seed_places[:5]


(79,
 ['Manhattan, New York, USA',
  'Dubai, UAE',
  'Tokyo, Japan',
  'Hong Kong',
  'Singapore'])

In [24]:
def location_search(search_query, category=None, language="en"):
    params = {"searchQuery": search_query, "language": language}
    if category:
        params["category"] = category
    j = ta_get("/location/search", params=params)
    return j.get("data", [])

def pick_best_geo_result(query, candidates):
    """
    Heuristic: prefer exact-ish name matches and non-zero/valid IDs.
    You can refine this later.
    """
    q = query.lower()
    scored = []
    for c in candidates:
        name = (c.get("name") or "").lower()
        loc_id = c.get("location_id")
        score = 0
        if loc_id: score += 5
        if name and (name in q or q in name): score += 10
        # slight preference if address info exists
        addr = c.get("address_obj") or {}
        if addr.get("country"): score += 1
        if addr.get("city"): score += 1
        scored.append((score, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

# Build dict of resolved places:
# top_places[location_id] = {"seed": "...", "name": "...", "address_string": "..."}
top_places = {}
unresolved = []

for place in seed_places:
    try:
        candidates = location_search(place, category="geos")
        best = pick_best_geo_result(place, candidates)
        if not best or not best.get("location_id"):
            unresolved.append({"seed": place, "reason": "no_best_match"})
            continue

        loc_id = best["location_id"]
        addr = best.get("address_obj") or {}
        top_places[loc_id] = {
            "seed": place,
            "name": best.get("name"),
            "address_string": addr.get("address_string"),
        }
    except Exception as e:
        unresolved.append({"seed": place, "reason": str(e)})

print("Resolved:", len(top_places))
print("Unresolved:", len(unresolved))
list(top_places.items())[:5]


Resolved: 78
Unresolved: 0


[('28953',
  {'seed': 'Manhattan, New York, USA',
   'name': 'New York',
   'address_string': 'NY'}),
 ('295424',
  {'seed': 'Dubai, UAE',
   'name': 'Dubai',
   'address_string': 'Dubai United Arab Emirates'}),
 ('298184',
  {'seed': 'Tokyo, Japan',
   'name': 'Tokyo',
   'address_string': 'Tokyo Prefecture'}),
 ('7917565',
  {'seed': 'Hong Kong',
   'name': 'Hong Kong Intl Airport',
   'address_string': '1 Sky Plaza Road, Hong Kong China'}),
 ('2146391',
  {'seed': 'Singapore',
   'name': 'Singapore River',
   'address_string': 'Singapore Singapore'})]

In [25]:
with open("top_places_resolved_ids.json", "w", encoding="utf-8") as f:
    json.dump(top_places, f, ensure_ascii=False, indent=2)

"top_places_resolved_ids.json"


'top_places_resolved_ids.json'

In [26]:
def location_details(location_id, language="en", currency="USD"):
    cache_path = CACHE_DIR / f"details_{location_id}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text("utf-8"))
    j = ta_get(f"/location/{location_id}/details", params={"language": language, "currency": currency})
    cache_path.write_text(json.dumps(j, ensure_ascii=False, indent=2), encoding="utf-8")
    return j

def nearby_search(lat, lon, category="attractions", radius=10, radius_unit="mi", language="en"):
    j = ta_get("/location/nearby_search", params={
        "latLong": f"{lat},{lon}",
        "category": category,
        "radius": radius,
        "radiusUnit": radius_unit,
        "language": language
    })
    return j.get("data", [])

RADIUS_MILES = 10  # change as needed

attractions_map = {}   # {attraction_id: {"name":..., "seed_place_id":..., "seed_place_name":...}}
seed_errors = []

for seed_geo_id, meta in top_places.items():
    try:
        geo = location_details(seed_geo_id)
        lat = geo.get("latitude")
        lon = geo.get("longitude")
        if lat is None or lon is None:
            seed_errors.append({"seed_geo_id": seed_geo_id, "seed": meta["seed"], "error": "missing lat/lon"})
            continue

        nearby = nearby_search(lat, lon, category="attractions", radius=RADIUS_MILES, radius_unit="mi")
        for item in nearby:
            aid = item.get("location_id")
            if not aid:
                continue
            if aid not in attractions_map:
                attractions_map[aid] = {
                    "name": item.get("name"),
                    "seed_geo_id": seed_geo_id,
                    "seed_place": meta["seed"],
                }
    except Exception as e:
        seed_errors.append({"seed_geo_id": seed_geo_id, "seed": meta["seed"], "error": str(e)})

print("Unique attractions found:", len(attractions_map))
print("Seed geo errors:", len(seed_errors))
list(attractions_map.items())[:5]


Unique attractions found: 742
Seed geo errors: 0


[('33344134',
  {'name': 'Leatherstocking Timber Products',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('9681910',
  {'name': 'C&C Taxi and Airport Transportation',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('19229164',
  {'name': 'Fortin Park',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('27041184',
  {'name': 'Caribbean Day Spa',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('2163160',
  {'name': 'Hanford Mills Museum',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'})]

In [27]:
with open("attractions_ids.json", "w", encoding="utf-8") as f:
    json.dump(attractions_map, f, ensure_ascii=False, indent=2)

"attractions_ids.json"


'attractions_ids.json'

In [28]:
def flatten_details(d):
    addr = d.get("address_obj") or {}
    return {
        "location_id": d.get("location_id"),
        "name": d.get("name"),
        "web_url": d.get("web_url"),
        "rating": d.get("rating"),
        "num_reviews": d.get("num_reviews"),
        "ranking_string": d.get("ranking_string"),
        "latitude": d.get("latitude"),
        "longitude": d.get("longitude"),
        "address": addr.get("address_string"),
        "city": addr.get("city"),
        "state": addr.get("state"),
        "country": addr.get("country"),
        "phone": d.get("phone"),
        "website": d.get("website"),
        # sometimes present for restaurants; usually not for attractions
        "price_level": d.get("price_level"),
    }

rows = []
detail_errors = []

for aid, meta in attractions_map.items():
    try:
        d = location_details(aid, language="en", currency="USD")
        row = flatten_details(d)
        row["seed_place"] = meta.get("seed_place")
        row["seed_geo_id"] = meta.get("seed_geo_id")
        rows.append(row)
    except Exception as e:
        detail_errors.append({"location_id": aid, "name": meta.get("name"), "error": str(e)})

df = pd.DataFrame(rows)

df.to_csv("attractions.csv", index=False)
df.to_json("attractions.json", orient="records", indent=2, force_ascii=False)

if detail_errors:
    pd.DataFrame(detail_errors).to_csv("attractions_errors.csv", index=False)

print("Saved attractions.csv and attractions.json")
print("Rows:", len(df), "Errors:", len(detail_errors))
df.head()


Saved attractions.csv and attractions.json
Rows: 742 Errors: 0


,location_id,name,web_url,rating,num_reviews,ranking_string,latitude,longitude,address,city,state,country,phone,website,price_level,seed_place,seed_geo_id
0,33344134,Leatherstocking Timber Products,https://www.tripadvisor.com/Attraction_Review-...,5.0,3,None,42.445614,-74.97493,"359 Delaware County Highway 11, West Oneonta, ...",West Oneonta,New York,United States,+1 607-436-9082,https://leatherstockinghandsplits.com/,None,"Manhattan, New York, USA",28953
1,9681910,C&C Taxi and Airport Transportation,https://www.tripadvisor.com/Attraction_Review-...,5.0,1,None,42.44465,-75.03002,"Oneonta, NY",Oneonta,New York,United States,+1 607-434-6531,https://www.facebook.com/candctransport13820/,None,"Manhattan, New York, USA",28953
2,19229164,Fortin Park,https://www.tripadvisor.com/Attraction_Review-...,5.0,2,None,42.453854,-75.013954,"101 Youngs Road, Oneonta, NY 13820",Oneonta,New York,United States,+1 607-432-2900,http://townofoneonta.org/town/parks-recreation...,None,"Manhattan, New York, USA",28953
3,27041184,Caribbean Day Spa,https://www.tripadvisor.com/Attraction_Review-...,None,0,None,42.447132,-75.0235,"5252 State Highway 23, West Oneonta, Oneonta, ...",West Oneonta,New York,United States,+1 607-435-7984,http://www.caribbeandayspa.biz,None,"Manhattan, New York, USA",28953
4,2163160,Hanford Mills Museum,https://www.tripadvisor.com/Attraction_Review-...,4.8,41,None,42.422653,-74.8864,"51 County Highway 12, Meredith, NY 13757-9998",Meredith,New York,United States,+1 607-278-5744,http://www.hanfordmills.org,None,"Manhattan, New York, USA",28953
